In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/brendan45774/test-file/tested.csv


In [3]:
# BEFORE CLEANING

# Load the original dataset again
titanic_df = pd.read_csv("/kaggle/input/datasets/brendan45774/test-file/tested.csv")

before_rows = titanic_df.shape[0]
before_duplicates = titanic_df.duplicated().sum()
before_nulls = titanic_df.isnull().sum().sum()

print(before_rows)
print(before_duplicates)
print(before_nulls)

418
0
414


In [4]:
# Load the dataset
titanic_df = pd.read_csv("/kaggle/input/datasets/brendan45774/test-file/tested.csv")

print("=" * 50)
print("DATA QUALITY REPORT")
print("=" * 50)

# Dataset shape
print("\nDataset Shape:")
print(titanic_df.shape)

# Missing values
print("\nMissing Values:")
print(titanic_df.isnull().sum())

# Duplicate rows
print("\nDuplicate Rows:")
print(titanic_df.duplicated().sum())

# Data types
print("\nData Types:")
print(titanic_df.dtypes)

# print(titanic_df[['Age', 'Fare', 'SibSp', 'Parch']].agg(['min', 'max']))

print("\nValue Range Anomalies")

# Age
print("Invalid Age values:",
      titanic_df[(titanic_df['Age'] < 0) | (titanic_df['Age'] > 120)].shape[0])

# Fare
print("Negative Fare values:",
      titanic_df[titanic_df['Fare'] < 0].shape[0])

# Pclass
print("Invalid Pclass values:",
      titanic_df[~titanic_df['Pclass'].isin([1, 2, 3])].shape[0])

# SibSp
print("Negative SibSp values:",
      titanic_df[titanic_df['SibSp'] < 0].shape[0])      #SibSp = Siblings + Spouses aboard

# Parch
print("Negative Parch values:",
      titanic_df[titanic_df['Parch'] < 0].shape[0])       #Parch = Parents + Children aboard

# Sex
print("Invalid Sex values:",
      titanic_df[~titanic_df['Sex'].isin(['male', 'female'])].shape[0])

# Embarked
print("Invalid Embarked values:",
      titanic_df[~titanic_df['Embarked'].isin(['C', 'Q', 'S'])].shape[0])

DATA QUALITY REPORT

Dataset Shape:
(418, 12)

Missing Values:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64

Duplicate Rows:
0

Data Types:
PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object

Value Range Anomalies
Invalid Age values: 0
Negative Fare values: 0
Invalid Pclass values: 0
Negative SibSp values: 0
Negative Parch values: 0
Invalid Sex values: 0
Invalid Embarked values: 0


In [5]:
# Missing data handling

# Age -> Median Imputation
titanic_df['Age'] = titanic_df['Age'].fillna(titanic_df['Age'].median())

# Fare -> Median Imputation
titanic_df['Fare'] = titanic_df['Fare'].fillna(titanic_df['Fare'].median())

# Cabin -> Drop column
titanic_df = titanic_df.drop(columns=['Cabin'], errors='ignore')

# Verify
print(titanic_df.isnull().sum())


PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64


# Missing Value Handling Strategy

### 1. Age → Median Imputation
- **Missing Values:** 86
- **Strategy:** Replace missing values with the **median age**.
- **Justification:** Age is a numerical feature and may contain outliers. The median is more robust to extreme values than the mean, making it a better choice for imputation.

### 2. Fare → Median Imputation
- **Missing Values:** 1
- **Strategy:** Replace the missing fare with the **median fare**.
- **Justification:** Fare is a numerical feature that is positively skewed due to some passengers paying very high fares. Median imputation preserves the central tendency without being affected by these extreme values.

### 3. Cabin → Row Deletion (or Column Deletion)
- **Missing Values:** 327 (about 78% of the dataset)
- **Strategy:** Drop the **Cabin** column.
- **Justification:** A large proportion of Cabin values are missing. Imputing such a high percentage would introduce unreliable information, so removing the column is the most appropriate choice.

### 4. Other Columns
- PassengerId, Survived, Pclass, Name, Sex, SibSp, Parch, Ticket, and Embarked have no missing values, so no imputation is required.

In [6]:
# Duplicate removal

# Check the number of duplicate rows
duplicate_rows = titanic_df.duplicated().sum()

print("Duplicate rows before removal:", duplicate_rows)

# Remove duplicate rows
titanic_df = titanic_df.drop_duplicates()

# Verify duplicates have been removed
print("Duplicate rows after removal:", titanic_df.duplicated().sum())

# Number of rows removed
print("Number of duplicate rows removed:", duplicate_rows)

Duplicate rows before removal: 0
Duplicate rows after removal: 0
Number of duplicate rows removed: 0



### Duplicate Row Removal

- Checked the dataset for duplicate rows using `duplicated().sum()`.
- Found **0 duplicate rows** in the dataset.
- Applied `drop_duplicates()` to remove duplicate records.
- After removal, verified that no duplicate rows remained.
- **Total duplicate rows removed:** 0.

In [7]:
# Standardize Gender column
titanic_df['Sex'] = titanic_df['Sex'].replace({
    'M': 'Male',
    'm': 'Male',
    'male': 'Male',
    'MALE': 'Male',
    'F': 'Female',
    'f': 'Female',
    'female': 'Female',
    'FEMALE': 'Female'
})

print(titanic_df['Sex'].unique())

['Male' 'Female']


### Standardisation

The `Sex` column was standardized by converting all values to title case
(e.g., `male` → `Male`, `female` → `Female`) to ensure consistent formatting.

The dataset does not contain any date-related columns; therefore, no date format
standardization to `datetime` was required.

In [8]:
# Outlier detection

# Select numeric columns
numeric_cols = titanic_df.select_dtypes(include=['int64', 'float64']).columns

for col in numeric_cols:
    Q1 = titanic_df[col].quantile(0.25)
    Q3 = titanic_df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = titanic_df[(titanic_df[col] < lower_bound) | (titanic_df[col] > upper_bound)]

    print(f"{col}: {len(outliers)} outliers")

PassengerId: 0 outliers
Survived: 0 outliers
Pclass: 0 outliers
Age: 36 outliers
SibSp: 11 outliers
Parch: 94 outliers
Fare: 55 outliers


### Outlier Detection

The IQR method was used to detect outliers in the numeric columns.

Outliers were identified in the following columns:
- Age: 36
- SibSp: 11
- Parch: 94
- Fare: 55

No outliers were removed or capped because these values represent valid passenger information rather than data entry errors. For example, higher fares correspond to first-class tickets, older passengers are genuine observations, and larger family sizes are possible. Therefore, all outliers were retained to preserve the original dataset.

In [9]:
# Data type correction

# Convert categorical columns
titanic_df["Pclass"] = titanic_df["Pclass"].astype("category")
titanic_df["Sex"] = titanic_df["Sex"].astype("category")
titanic_df["Embarked"] = titanic_df["Embarked"].astype("category")

# PassengerId should remain as string/object
titanic_df["PassengerId"] = titanic_df["PassengerId"].astype(str)

# Verify the changes
titanic_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   PassengerId  418 non-null    object  
 1   Survived     418 non-null    int64   
 2   Pclass       418 non-null    category
 3   Name         418 non-null    object  
 4   Sex          418 non-null    category
 5   Age          418 non-null    float64 
 6   SibSp        418 non-null    int64   
 7   Parch        418 non-null    int64   
 8   Ticket       418 non-null    object  
 9   Fare         418 non-null    float64 
 10  Embarked     418 non-null    category
dtypes: category(3), float64(2), int64(3), object(3)
memory usage: 27.9+ KB


__Observation__

Data type correction was performed to ensure that each column had an appropriate data type. PassengerId, Name, Ticket, and Cabin were retained as object types because they contain textual or identifier information. Pclass, Sex, and Embarked were converted to the category data type since they represent categorical variables with a limited number of unique values, improving memory efficiency. Age and Fare were retained as float64 because they contain decimal values, and Age also includes missing values. The count-based variables (Survived, SibSp, and Parch) were kept as int64 because they represent whole numbers. Overall, all columns now have suitable data types for analysis and modeling.

In [11]:
# After cleaning

after_rows = titanic_df.shape[0]
after_duplicates = titanic_df.duplicated().sum()
after_nulls = titanic_df.isnull().sum().sum()
after_dtypes = titanic_df.dtypes

In [12]:
import pandas as pd

summary = pd.DataFrame({
    "Metric": [
        "Row Count",
        "Duplicate Rows",
        "Total Missing Values",
        "Correct Data Types"
    ],
    "Before Cleaning": [418, 0, 414, "9/12"],
    "After Cleaning": [418, 0, 0, "11/11"]
})

summary

,Metric,Before Cleaning,After Cleaning
0,Row Count,418,418
1,Duplicate Rows,0,0
2,Total Missing Values,414,0
3,Correct Data Types,9/12,11/11


Why these values?

Row Count: 418 → 418
    No duplicate rows were found, so no rows were removed.
  
Duplicate Rows: 0 → 0
    Your dataset had no duplicate rows before cleaning.
  
Total Missing Values: 414 → 0

    Age: 86 missing → filled with median.
    Fare: 1 missing → filled with median.
    Cabin: 327 missing → column dropped.
    Total missing values after cleaning = 0.
    
Correct Data Types: 9/12 → 11/11

    Before cleaning, Pclass, Sex, and Embarked were not in their optimal category type.
    After cleaning, Cabin was dropped, leaving 11 columns, and all remaining columns have appropriate data types.

In [13]:
# Save the cleaned dataset to a new CSV file
titanic_df.to_csv("titanic_cleaned.csv", index=False)

print("Cleaned dataset saved successfully as 'titanic_cleaned.csv'")

Cleaned dataset saved successfully as 'titanic_cleaned.csv'


__OBSERVATION__

The cleaned dataset was saved as a new CSV file (titanic_cleaned.csv) using the to_csv() function. The index=False parameter was used to exclude the DataFrame index from the saved file, ensuring a clean and well-formatted dataset for future analysis or modeling.